In [78]:
import sdmx
import time
import numpy as np
import pendulum

In [79]:
pendulum.now().timezone

Timezone('Europe/Istanbul')

In [2]:
sdmx.add_source({
    "id": "TUIK",
    "url": "https://nsiws.tuik.gov.tr/rest",
    "name": "Türkiye İstatistik Kurumu (TÜİK)"
}, override=True)

In [ ]:
df = 'DF_YDUFE_EDO'
tuik = sdmx.Client('TUIK')
print("START META", df)
start_m = time.perf_counter()
flow = tuik.dataflow(df, agency_id='TR', detail='full', references='descendants')
end_m = time.perf_counter()
print("END META", "elapsed", "%.3f"%(end_m-start_m), "sec")
dimensions = {comp.id: comp.concept_identity.id 
            for struct in flow.structure.items()
            for comp in struct[1].dimensions.components}
measures = {msr.id: msr.concept_identity.id 
            for struct in flow.structure.items()
            for msr in struct[1].measures}
metadata = dict(dimensions, **measures)
codelists = {comp.id: {k: v.name["tr"] for k, v in comp.local_representation.enumerated.items.items()}
            for struct in flow.structure.items()
            for comp in struct[1].dimensions.components if comp.local_representation.enumerated}
concepts = {dim: [c for csch in flow.concept_scheme.values()
                for c in csch.items.values() if c.id == cid][0].name["tr"]
            for dim, cid in metadata.items()}
print("START SLEEP", "10 sec")
time.sleep(10)
print("END SLEEP")
print("START DATA", df)
start_d = time.perf_counter()
data = tuik.data(df)
end_d = time.perf_counter()
print("END DATA", "elapsed", "%.3f"%(end_d-start_d), "sec")
pd_data = sdmx.to_pandas(data)
for cl in codelists.keys():
    repeated, control = {}, {}
    all_index = [(val, codelists[vals.name][val])
        for vals in pd_data.index.levels 
        for val in list(vals) if vals.name == cl
    ]
    for idxs in all_index:
        if idxs[1] in control.keys():
            if idxs[1] in repeated.keys():
                repeated[idxs[1]].append(idxs[0])
            else:
                repeated[idxs[1]] = [control[idxs[1]], idxs[0]]
        else:
            control[idxs[1]] = idxs[0]
    for _, idxs in repeated.items():
        pd_data.index
    pd_data.index = pd_data.index.set_levels([codelists[vals.name][val] 
        for vals in pd_data.index.levels 
        for val in list(vals) if vals.name == cl
    ], level=cl)
pd_data.rename(concepts["OBS_VALUE"], inplace=True)
pd_data.index.set_names([concepts[name] for name in pd_data.index.names], inplace=True)
# pd_data.to_csv(f'/opt/airflow/files/{re.sub(r"^DF_", "", df)}.csv')

START META DF_YDUFE_EDO
END META elapsed 6.165 sec
START SLEEP 10 sec
END SLEEP
START DATA DF_YDUFE_EDO
END DATA elapsed 21.026 sec


ValueError: Level values must be unique: ['Madencilik ve taş ocakçılığı', 'Metal cevherleri madenciliği', 'Demir cevherleri madenciliği', 'Demir cevherleri madenciliği', 'Demir dışı metal cevherleri madenciliği', 'Diğer demir dışı metal cevherleri madenciliği', 'Diğer madencilik ve taş ocakçılığı', 'Kum, kil ve taş ocakçılığı', 'Süsleme ve yapı taşları ile kireç taşı, alçı taşı, tebeşir ve kayağantaşı (arduvaz-kayraktaşı) ocakçılığı', 'Çakıl ve kum ocaklarının faaliyetleri; kil ve kaolin çıkarımı', 'Başka yerde sınıflandırılmamış madencilik ve taş ocakçılığı', 'Kimyasal ve gübreleme amaçlı mineral madenciliği', 'Başka yerde sınıflandırılmamış diğer madencilik ve taş ocakçılığı', 'Madencilik ve taş ocakçılığı, imalat', 'İmalat', 'Gıda ürünlerinin imalatı', 'Etin işlenmesi ve saklanması ile et ürünlerinin imalatı', 'Etin işlenmesi ve saklanması', 'Kümes hayvanları etlerinin işlenmesi ve saklanması', 'Et ve kümes hayvanları etlerinden üretilen ürünlerin imalatı', 'Balık, kabuklu deniz hay

In [42]:
data = tuik.data(df)
pd_data = sdmx.to_pandas(data)

In [83]:
pd_data["2019-01", "TR"]

/var/folders/_5/m1gpx_mx6156_88k1hvfpkzm0000gn/T/ipykernel_75317/3412195176.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  pd_data["2019-01", "TR"]


INDICATOR  DEGISIM  BASE_PER  FREQ  FIYAT_ENDEKS_KAPSAM  MALIYET_GRUP  FAALIYET_CPA_2008  FAALIYET_NACE_REV2  FAALIYET_CPA_2_1  FAAL_GRUP
F_YDUFE    1        2010      M     _Z                   _Z            _Z                 B                   _Z                3            300.10
                                                                                          B_C                 _Z                _T           300.20
                                                                                          B07                 _Z                4            275.13
                                                                                          B071                _Z                5            103.32
                                                                                          B0710               _Z                6               NaN
                                                                                                                          

In [ ]:
times = list(pd_data.index.get_level_values("TIME_PERIOD"))
times.sort(reverse=True)
times

['2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '2026-02',
 '20

: 

In [38]:
pd_data.xs("B0710", level='FAALIYET_NACE_REV2')

TIME_PERIOD  REF_AREA  INDICATOR  DEGISIM  BASE_PER  FREQ  FIYAT_ENDEKS_KAPSAM  MALIYET_GRUP  FAALIYET_CPA_2008  FAALIYET_CPA_2_1  FAAL_GRUP
2019-01      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 _Z                6           NaN
2019-02      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 _Z                6           NaN
2019-03      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 _Z                6           NaN
2019-04      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 _Z                6           NaN
2019-05      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 _Z                6           NaN
                                                                                                                            

In [33]:
[level for level in pd_data.index.levels if level.name == 'FAALIYET_NACE_REV2'][0]

Index(['B', 'B07', 'B071', 'B0710', 'B072', 'B0729', 'B08', 'B081', 'B0811',
       'B0812',
       ...
       'C325', 'C3250', 'C329', 'C3291', 'C3299', 'MIG_CAG', 'MIG_DCOG',
       'MIG_ING', 'MIG_NDCOG', 'MIG_NRG'],
      dtype='object', name='FAALIYET_NACE_REV2', length=305)

In [41]:
pd_data.index.get_level_values('FAALIYET_NACE_REV2')

Index(['B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       ...
       'MIG_NRG', 'MIG_NRG', 'MIG_NRG', 'MIG_NRG', 'MIG_NRG', 'MIG_NRG',
       'MIG_NRG', 'MIG_NRG', 'MIG_NRG', 'MIG_NRG'],
      dtype='object', name='FAALIYET_NACE_REV2', length=257065)

In [77]:
pd_data.shape[0]

257065

In [40]:
pd_data.to_csv('ydufe.csv')

In [ ]:
test = [codelists[vals.name][val] 
    for vals in pd_data.index.levels 
    for val in list(vals) if vals.name == 'FAALIYET_NACE_REV2'
]

['Madencilik ve taş ocakçılığı',
 'Metal cevherleri madenciliği',
 'Demir cevherleri madenciliği',
 'Demir cevherleri madenciliği',
 'Demir dışı metal cevherleri madenciliği',
 'Diğer demir dışı metal cevherleri madenciliği',
 'Diğer madencilik ve taş ocakçılığı',
 'Kum, kil ve taş ocakçılığı',
 'Süsleme ve yapı taşları ile kireç taşı, alçı taşı, tebeşir ve kayağantaşı (arduvaz-kayraktaşı) ocakçılığı',
 'Çakıl ve kum ocaklarının faaliyetleri; kil ve kaolin çıkarımı',
 'Başka yerde sınıflandırılmamış madencilik ve taş ocakçılığı',
 'Kimyasal ve gübreleme amaçlı mineral madenciliği',
 'Başka yerde sınıflandırılmamış diğer madencilik ve taş ocakçılığı',
 'Madencilik ve taş ocakçılığı, imalat',
 'İmalat',
 'Gıda ürünlerinin imalatı',
 'Etin işlenmesi ve saklanması ile et ürünlerinin imalatı',
 'Etin işlenmesi ve saklanması',
 'Kümes hayvanları etlerinin işlenmesi ve saklanması',
 'Et ve kümes hayvanları etlerinden üretilen ürünlerin imalatı',
 'Balık, kabuklu deniz hayvanları ve yumuşakçal

In [ ]:
repeated, control = {}, {}
all_index = [(val, codelists[vals.name][val])
    for vals in pd_data.index.levels 
    for val in list(vals) if vals.name == cl
]
for idxs in all_index:
    if idxs[1] in control.keys():
        if idxs[1] in repeated.keys():
            repeated[idxs[1]].append(idxs[0])
        else:
            repeated[idxs[1]] = [control[idxs[1]], idxs[0]]
    else:
        control[idxs[1]] = idxs[0]
for _, idxs in repeated.items():
    orig, rest = idxs[0], idxs[1:]
    origdata = pd_data.xs(orig, level=cl)
    for r in rest:
        restdata = pd_data.xs(r, level=cl)
        if (origdata.values == restdata.values).all() or (np.isnan(restdata.values)).all():
            
        print(orig, len(origdata.values), r, len(restdata.values), (origdata.values == restdata.values).all(), (np.isnan(restdata.values)).all())

B071 382 B0710 382 True False True
C102 922 C1020 922 False False False
C11 922 C110 922 False True False
C12 922 C120 922 False True False
C12 922 C1200 922 False True False
C131 922 C1310 922 False True False
C132 922 C1320 922 False True False
C133 922 C1330 922 False True False
C142 922 C1420 922 False True False
C152 922 C1520 922 False True False
C161 922 C1610 922 False True False
C191 922 C1910 922 True False True
C192 922 C1920 922 False True False
C202 922 C2020 922 False True False
C203 922 C2030 922 False True False
C206 922 C2060 922 False True False
C212 922 C2120 922 False True False
C232 922 C2320 922 False True False
C237 922 C2370 922 False True False
C241 922 C2410 922 False True False
C242 922 C2420 922 False True False
C255 922 C2550 922 False True False
C262 518 C2620 518 True False True
C263 922 C2630 922 True False True
C264 922 C2640 922 True False True
C272 922 C2720 922 False True False
C274 922 C2740 922 False True False
C279 442 C2790 442 False True False
C

In [58]:
origdata = pd_data.xs('B071', level=cl)
restdata = pd_data.xs('B0710', level=cl)   

In [ ]:
import pandas as pd

In [73]:
np.isnan(restdata.values[0])

np.True_

In [16]:
repeated

{'Demir cevherleri madenciliği': ['B071', 'B0710'],
 'Balık, kabuklu deniz hayvanları ve yumuşakçaların işlenmesi ve saklanması': ['C102',
  'C1020'],
 'İçeceklerin imalatı': ['C11', 'C110'],
 'Tütün ürünleri imalatı': ['C12', 'C120', 'C1200'],
 'Tekstil elyafının hazırlanması ve bükülmesi': ['C131', 'C1310'],
 'Dokuma': ['C132', 'C1320'],
 'Tekstil ürünlerinin bitirilmesi': ['C133', 'C1330'],
 'Kürkten eşya imalatı': ['C142', 'C1420'],
 'Ayakkabı, bot, terlik vb. imalatı': ['C152', 'C1520'],
 'Ağaçların biçilmesi ve planyalanması': ['C161', 'C1610'],
 'Kok fırını ürünlerinin imalatı': ['C191', 'C1910'],
 'Rafine edilmiş petrol ürünleri imalatı': ['C192', 'C1920'],
 'Haşere ilaçları ve diğer zirai-kimyasal ürünlerin imalatı': ['C202',
  'C2020'],
 'Boya, vernik ve benzeri kaplayıcı maddeler ile matbaa mürekkebi ve macun imalatı': ['C203',
  'C2030'],
 'Suni veya sentetik elyaf imalatı': ['C206', 'C2060'],
 'Eczacılığa ilişkin ilaçların imalatı': ['C212', 'C2120'],
 'Ateşe dayanıklı (re

In [5]:
pd_data

TIME_PERIOD  REF_AREA  INDICATOR  DEGISIM  BASE_PER  FREQ  FIYAT_ENDEKS_KAPSAM  MALIYET_GRUP  FAALIYET_CPA_2008  FAALIYET_NACE_REV2  FAALIYET_CPA_2_1  FAAL_GRUP
2010-01      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 B                   _Z                3             98.95
2010-02      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 B                   _Z                3             96.26
2010-03      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 B                   _Z                3             97.40
2010-04      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 B                   _Z                3             94.52
2010-05      TR        F_YDUFE    1        2010      M     _Z                   _Z            _Z                 B                   _Z                3   

In [127]:
dimensions_ = {comp.id: comp.concept_identity.id 
            for struct in flow.structure.items()
            for comp in struct[1].dimensions.components}
measures_ = {msr.id: msr.concept_identity.id 
            for struct in flow.structure.items()
            for msr in struct[1].measures}
metadata = dict(dimensions_, **measures_)
codelists_ = {comp.id: {k: v.name["tr"] for k, v in comp.local_representation.enumerated.items.items()}
            for struct in flow.structure.items()
            for comp in struct[1].dimensions.components if comp.local_representation.enumerated}
# concepts_ = {[k for k, v in dimensions_.items() if v == concept.id][0]: concept.name["tr"]
#             for csch in flow.concept_scheme.values()
#             for concept in csch.items.values() if concept.id in metadata.values()}
concepts_ = {dim: [c for csch in flow.concept_scheme.values()
                for c in csch.items.values() if c.id == cid][0].name["tr"]
            for dim, cid in metadata.items()}

In [16]:
len(data.data[0])

257065

In [65]:
len(data.data[0].obs)

257065

In [15]:
len(data.data[0].series)

1515

In [63]:
sum = 0
for serie, obs in data.data[0].series.items():
    sum += len(obs)

In [92]:
data.data[0].obs[0]

Observation(attached_attribute={}, series_key=<SeriesKey: REF_AREA=TR, INDICATOR=F_YDUFE, DEGISIM=1, BASE_PER=2010, FREQ=M, FIYAT_ENDEKS_KAPSAM=_Z, MALIYET_GRUP=_Z, FAALIYET_CPA_2008=_Z, FAALIYET_NACE_REV2=B, FAALIYET_CPA_2_1=_Z, FAAL_GRUP=3>, dimension=<Key: TIME_PERIOD=2010-01>, value='98.95', group_keys=set(), value_for=None)

In [64]:
sum

257065

In [67]:
[comp.concept_identity.id
for struct in flow.structure.items()
for comp in struct[1].dimensions.components]

['REF_AREA',
 'INDICATOR',
 'DEGISIM',
 'BASE_PER',
 'FREQ',
 'FIYAT_ENDEKS_KAPSAM',
 'MAALIYET_GRUP',
 'FAALIYETCPA_2008',
 'FAALIYET_NACE_REV2',
 'FAALIYETCPA_2_1',
 'FAAL_GRUP',
 'TIME_PERIOD']

In [ ]:
len(flow.codelist)

1247

In [80]:
with open("yo.txt", "w") as f:
    f.write('\n'.join(list(flow.codelist.keys())))

In [ ]:
flow.codelist["CL_MALIYET_GRUP"]

<Codelist TR:CL_MALIYET_GRUP(1.0) (4 items): Cost Groups Code List (PPI)>

In [84]:
yo = {concept.id: concept
for csch in flow.concept_scheme.values()
for concept in csch.items.values() if concept.id in dimensions or concept.id in measures}

In [ ]:
yo["MALIYET_GRUP"]

KeyError: 'MALIYET_GRUP'

In [103]:
[comp.id
for struct in flow.structure.items()
for comp in struct[1].dimensions.components]

['REF_AREA',
 'INDICATOR',
 'DEGISIM',
 'BASE_PER',
 'FREQ',
 'FIYAT_ENDEKS_KAPSAM',
 'MALIYET_GRUP',
 'FAALIYET_CPA_2008',
 'FAALIYET_NACE_REV2',
 'FAALIYET_CPA_2_1',
 'FAAL_GRUP',
 'TIME_PERIOD']

In [4]:
pd_data

TIME_PERIOD  REF_AREA  INDICATOR                        DEGISIM                                    BASE_PER  FREQ   FIYAT_ENDEKS_KAPSAM  MALIYET_GRUP  FAALIYET_CPA_2008  FAALIYET_NACE_REV2  FAALIYET_CPA_2_1  FAAL_GRUP
2010-01      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                     2010      Aylık  Uygulanabilir değil  _Z            _Z                 B                   _Z                3             98.95
2010-02      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                     2010      Aylık  Uygulanabilir değil  _Z            _Z                 B                   _Z                3             96.26
2010-03      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                     2010      Aylık  Uygulanabilir değil  _Z            _Z                 B                   _Z                3             97.40
2010-04      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                    

In [ ]:
pd_data

Zaman    Gözlem Sıklığı  Ölçü Birimi  Dış Ticaret Göstergeleri         Dış Ticaret Türü  İstatistiksel Gösterge  Endeks-Değişim       İktisadi Faaliyet                                     Dış Ticaret Endeks Sektör Kodları  Temel Dönem/Yıl  Coğrafi Kapsam  Düzeltme           
2015     Yıllık          Oran         Altın Hariç Dış Ticaret Hadleri  Diğer             Dış Ticaret Endeksleri  Uygulanabilir değil  -                                                     0+1 Gıda, içecek ve tütün          2015             Türkiye         Uygulanabilir değil    100.0
2016     Yıllık          Oran         Altın Hariç Dış Ticaret Hadleri  Diğer             Dış Ticaret Endeksleri  Uygulanabilir değil  -                                                     0+1 Gıda, içecek ve tütün          2015             Türkiye         Uygulanabilir değil     95.5
2017     Yıllık          Oran         Altın Hariç Dış Ticaret Hadleri  Diğer             Dış Ticaret Endeksleri  Uygulanabilir değil  -                   